In [1]:
import sys
sys.path.append("..")

In [4]:
import tqdm
import os
import time
import warnings
import pickle
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import cvxpy as cp
from copy import deepcopy

from src.data import *
from src.model import *
from src.recourse import *
from src.utils import *

warnings.filterwarnings('ignore')

In [3]:
def append_result(d, algorithm, seed, alpha, lamb, i, x_0, theta_0, x_r, theta_r=None):
    d["algorithm"].append(algorithm)
    d["seed"].append(seed)
    d["alpha"].append(alpha)
    d["lambda"].append(lamb)
    d["i"].append(i)
    d["x_0"].append(x_0.round(4))
    d["x_r"].append(x_r.round(4))
    d["theta_0"].append(theta_0.round(4))

def append_result_time(d_time, index, e_time):
    d_time["i"].append(index)
    d_time["e_time"].append(np.float64(e_time).round(5))

In [13]:
def run_experiment(dataset: Dataset, params: dict):
    for seed in params['seeds']:
        (train_data, test_data), (train_data_shift, test_data_shift) = dataset.get_data(seed, shift=True)
        X_train_shift, y_train_shift = train_data_shift
        
        if params["base_model"] == "lr":
            base_model = LR()
        elif params["base_model"] == "nn":
            base_model = NN(X_train_shift.shape[1])
        base_model.train(X_train_shift.values, y_train_shift.values)

        f_name = f"../results/recourse_model/{params['base_model']}_shift_{dataset.name}_{seed}.pkl"
        with open(f_name, "wb") as f:
            pickle.dump(base_model, f)    

In [20]:
params = {}
params["base_model"] = "nn"
params["seeds"] = range(5)

datasets = [GermanDataset()]

for dataset in datasets:
    run_experiment(dataset, params)

In [17]:
seed = 0
f_name = f"../results/recourse_model/{params['base_model']}_shift_{dataset.name}_{seed}.pkl"
with open(f_name, "rb") as f:
    model = pickle.load(f)
    print(model.model.coef_)

[[-0.41605165 -0.06491737  0.11525762 -0.20249737 -0.21135741  0.29077308
   0.12280268]]
